In [1]:
%load_ext autoreload
%autoreload 2


# Import Libraries

In [11]:
import os

import pandas as pd
from catboost import Pool, CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV

# Read Data

In [29]:
path = "../"
train_data = pd.read_csv(os.path.join(path, "data/train.csv"))
test_data = pd.read_csv(os.path.join(path, "data/test.csv"))
print(f"Number of rows and columns in the train data set: {train_data.shape}")
print(f"Number of rows and columns in the test data set: {test_data.shape}")
train_data.head()

Number of rows and columns in the train data set: (48665, 2)
Number of rows and columns in the test data set: (12167, 2)


,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [17]:
train_data.groupby("rate").describe()

text                               
      count unique                top freq
rate                                      
1      4138   4130             Грязно    3
2      2410   2407  Отстойный магазин    2
3      6126   6070          Нормально    7
4      9922   9763               Норм   14
5     26069  24804    Хороший магазин  107

# Preparing the data and creating Catboost model

In [30]:
X = train_data["text"]
y = train_data["rate"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X_test = test_data["text"]

In [31]:
model = CatBoostClassifier(
    random_seed=42
)

In [32]:
param_grid = {
    'iterations': [100, 200], 
    'depth': [6, 8, 10, 15], 
    'learning_rate': [0.05, 0.1, 0.2],  
    'auto_class_weights': [None, 'Balanced'],
    'loss_function': ['MultiClass'], 
    'eval_metric': ['TotalF1'], 
}

In [33]:
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=4, verbose=10)
grid_search.fit(
    X=X_train.to_frame(),
    y=y_train,
    **{
        'text_features': [0],
        'eval_set': (X_val.to_frame(), y_val)
    }
)

print("Best params: ", grid_search.best_params_)
print("Best score: ", grid_search.best_score_)

Fitting 4 folds for each of 48 candidates, totalling 192 fits
[CV 1/4; 1/48] START auto_class_weights=None, depth=6, eval_metric=TotalF1, iterations=100, learning_rate=0.05, loss_function=MultiClass
[CV 1/4; 1/48] END auto_class_weights=None, depth=6, eval_metric=TotalF1, iterations=100, learning_rate=0.05, loss_function=MultiClass;, score=nan total time=   0.0s
[CV 2/4; 1/48] START auto_class_weights=None, depth=6, eval_metric=TotalF1, iterations=100, learning_rate=0.05, loss_function=MultiClass
[CV 2/4; 1/48] END auto_class_weights=None, depth=6, eval_metric=TotalF1, iterations=100, learning_rate=0.05, loss_function=MultiClass;, score=nan total time=   0.0s
[CV 3/4; 1/48] START auto_class_weights=None, depth=6, eval_metric=TotalF1, iterations=100, learning_rate=0.05, loss_function=MultiClass
[CV 3/4; 1/48] END auto_class_weights=None, depth=6, eval_metric=TotalF1, iterations=100, learning_rate=0.05, loss_function=MultiClass;, score=nan total time=   0.0s
[CV 4/4; 1/48] START auto_cla

KeyboardInterrupt: 

In [ ]:
best_model = grid_search.best_estimator_
best_model.save_model("best_model.cbm")

In [ ]:
model = CatBoostClassifier()
model.load_model("best_model.cbm")

# Predict

In [5]:
# Preparing data in Pool format
dataset_test = Pool(
    data=X_test,
    text_features=[0]
)
predict_classes = model.predict(dataset_test)
predictions = predict_classes

# Create submission

In [6]:
submission = pd.read_csv(os.path.join(path, "data/sample_submission.csv"))
submission["rate"] = predictions
submission.head()

,index,rate
0,0,5
1,1,5
2,2,5
3,3,4
4,4,5


In [7]:
submission.to_csv("submission.csv", index=False)

In [8]:
submission.groupby("rate").describe()

NameError: name 'submission' is not defined